# Task 2 — Full Comparison with Resume and Checkpoint Support


This notebook reproduces all five Task 2 comparison tables from the
review report:

- conversation vs non-conversation;
- conversation vs co-building;
- conversation vs co-merging;
- co-merging vs co-building;
- three-class activity recognition.

Every subtask is evaluated across all seven sensor combinations under
the same three regimes used in the report.


## Output convention

The notebooks report two distinct result types:

- **Pooled metrics**, matching the historical report: all held-out
  predictions are combined before accuracy and macro-F1 are calculated.
- **Fold mean ± SD**, measuring variation across held-out groups.

The best configuration in each regime is selected by pooled macro-F1,
exactly as in the report. The fold mean and SD are then attached as
additional publication statistics.

With the default report-reproduction settings:

- Classical: 5 tasks × 7 sensor combinations × 2 time conditions ×
  2 models × 2 k-values = **280 configurations**, each under LOGO.
- DL: 5 tasks × 7 sensor combinations × 4 architectures × 1 seed =
  **140 LOGO runs**.

This is computationally substantial. Partial CSV checkpoints are
written after the DL runs so interrupted Colab sessions retain the
completed summaries.

In [1]:
# Optional Google Drive mount for Google Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except (ImportError, ModuleNotFoundError):
    print(
        "Not running in Colab. Update DATA_ROOT in the "
        "configuration cell."
    )


Mounted at /content/drive


In [2]:
# ================================================================
# SETUP
# ================================================================

import os
import re
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [3]:
# ================================================================
# CONFIG
# ================================================================

DATA_ROOT = "/content/drive/MyDrive/thesis/data"

# Original feature tables used for the report.
ACTIVITY_DATA_PATH = (
    f"{DATA_ROOT}/INTERACTION_ABLATIONS/"
    "activity3_advanced_merged_10s_features.csv"
)

INTERACTION_DATA_PATHS = [
    (
        f"{DATA_ROOT}/INTERACTION_BINARY_5S_SPECIALIZED_OE/"
        "binary_5s_specialized_oe_merged_all_features.csv"
    ),
    (
        f"{DATA_ROOT}/INTERACTION_BINARY_5S_ADVANCED_FEATURES/"
        "binary_5s_all_sensor_advanced_features.csv"
    ),
]

OUT_DIR = f"{DATA_ROOT}/PUBLICATION_TASK2_FULL_COMPARISON"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_TASKS = [
    "conversation_vs_nonconversation",
    "conversation_vs_building",
    "conversation_vs_merging",
    "merging_vs_building",
    "three_class_activity",
]

SENSOR_COMBINATIONS = {
    "OE": ["OE"],
    "OPTI": ["OPTI"],
    "XSENS": ["XSENS"],
    "OE_OPTI": ["OE", "OPTI"],
    "OE_XSENS": ["OE", "XSENS"],
    "OPTI_XSENS": ["OPTI", "XSENS"],
    "OE_OPTI_XSENS": ["OE", "OPTI", "XSENS"],
}

# Keep this True. Completed CSV files are reloaded automatically after
# a Colab disconnect, and already completed DL configurations are skipped.
RESUME_EXISTING = True

RUN_CLASSICAL = True
RUN_DL = True

# ----------------------------------------------------------------
# REPORT REPRODUCTION SETTINGS
# ----------------------------------------------------------------
# The report's full seven-combination classical table was produced
# from Logistic Regression and Linear SVC at k=80 and k=200.
REPORT_REPRODUCTION_MODE = True

# Set REPORT_REPRODUCTION_MODE=False to run the larger exploratory
# grid: RBF-SVC, Random Forest and Extra Trees, plus more k values.
K_CLASSICAL_FULL = [40, 80, 120, 200, "all"]
TIME_CONDITIONS = ["no_elapsed", "with_elapsed"]

# DL comparison: no elapsed time only.
# All four sequence architectures are screened because different
# models win for different tasks/sensor combinations in the report.
RUN_DL_MODEL_TYPES = ["lstm", "bilstm", "gru", "transformer"]
K_DL = [120]

# Seed 42 reproduces the historical report comparison.
# Adding seeds, e.g. [42, 7, 21], additionally enables seed-level SD.
SEEDS = [42]

FAST_MAX_EPOCHS = 25
FAST_PATIENCE = 4
FAST_BATCH_SIZE = 256
FAST_PRED_BATCH_SIZE = 1024

# Full leave-one-group-out evaluation.
MAX_LOGO_FOLDS = None

# Row-level predictions are not needed for mean/SD and create large files.
SAVE_CLASSICAL_PREDICTIONS = False
SAVE_FAST_DL_PREDICTIONS = False

RANDOM_STATE = 42


## Data and feature preparation

This section uses the same feature-detection and task-construction
logic as the original all-sensor ablation notebook. Elapsed time is
appended only to the classical `with_elapsed` condition. It is never
included in the DL comparison.

In [4]:
# ================================================================
# GENERAL HELPERS
# ================================================================

def safe_name(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def unique_feats(feats):
    return list(dict.fromkeys(feats))


def find_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None


def infer_window_seconds(df, start_col, end_col, default):
    if start_col in df.columns and end_col in df.columns:
        dur = pd.to_numeric(df[end_col], errors="coerce") - pd.to_numeric(df[start_col], errors="coerce")
        val = float(np.nanmedian(dur))
        if np.isfinite(val) and val > 0:
            return val
    return float(default)


def add_elapsed_min(df, group_col, start_col, end_col):
    df = df.copy()
    if "elapsed_min" in df.columns:
        df["elapsed_min"] = pd.to_numeric(df["elapsed_min"], errors="coerce")
        return df

    if start_col not in df.columns:
        raise ValueError(f"Cannot create elapsed_min because {start_col} is missing.")

    if end_col in df.columns:
        df["window_mid"] = (
            pd.to_numeric(df[start_col], errors="coerce") +
            pd.to_numeric(df[end_col], errors="coerce")
        ) / 2.0
    else:
        df["window_mid"] = pd.to_numeric(df[start_col], errors="coerce")

    df["elapsed_min"] = (df["window_mid"] - df.groupby(group_col)["window_mid"].transform("min")) / 60.0
    return df


BAD_TOKENS = [
    "label", "target", "class", "group", "window", "time", "elapsed",
    "pred", "prediction", "correct", "fold", "split", "index"
]


def looks_bad_feature_name(c):
    cl = str(c).lower()
    return any(tok in cl for tok in BAD_TOKENS)


def is_oe_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False
    return (
        c.startswith("oe__")
        or c.startswith("oe_")
        or c.startswith("oebest__")
        or c.startswith("ear_")
        or c.startswith("mag__")
        or c.startswith("mag_")
    )


def is_opti_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False

    # Current advanced merged datasets normally use opti2_ / opti2__ prefixes.
    if c.startswith("opti2_") or c.startswith("opti2__") or c.startswith("opti__") or c.startswith("opti_"):
        return True

    # Fallback for older unprefixed OptiTrack feature names.
    # Kept conservative to avoid catching OE/XSens features.
    fallback_tokens = [
        "dist_close", "dist_mid", "dist_far", "dist_disp",
        "centroid", "spread", "triangle", "perimeter", "compactness",
        "nearest", "farthest", "pairdist", "pair_dist",
        "approach", "separation", "proximity", "relative_pos",
    ]
    if any(tok in cl for tok in fallback_tokens):
        return True

    # Some old OptiTrack columns used simple speed names.
    if cl in {"speed_min", "speed_mid", "speed_max", "centroid_speed"}:
        return True

    return False


def is_xsens_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False
    return (
        c.startswith("xsens2__")
        or c.startswith("xsens2_")
        or c.startswith("xsens__")
        or c.startswith("xsens_")
    )


def clean_feature_list(dataframe, feats, label_cols=None, include_elapsed=False):
    label_cols = set(label_cols or [])
    bad_cols = set(label_cols) | {
        "group", "window_start", "window_end", "window_mid", "video_time_s", "time_s",
        "label", "target", "class", "activity", "activity_class", "general_class",
        "binary_label", "recognition_label", "task_label", "pred", "prediction", "correct",
    }

    cleaned = []
    for f in unique_feats(feats):
        if f not in dataframe.columns:
            continue
        if f in bad_cols:
            continue

        fl = str(f).lower()
        if f == "elapsed_min":
            if include_elapsed:
                cleaned.append(f)
            continue

        if not include_elapsed and "elapsed" in fl:
            continue
        if any(tok in fl for tok in ["label", "target", "pred", "prediction", "correct"]):
            continue

        x = pd.to_numeric(dataframe[f], errors="coerce").values
        if np.isfinite(x).sum() < 20:
            continue
        if np.nanstd(x) < 1e-12:
            continue
        cleaned.append(f)

    return unique_feats(cleaned)


def make_classical_models(n_classes):
    return {
        "logreg_C1": LogisticRegression(
            C=1.0,
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear" if n_classes == 2 else "lbfgs",
            multi_class="auto",
            random_state=RANDOM_STATE,
        ),
        "linearSVC_C1": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=10000,
            dual=False,
        ),
        "rbfSVC_C1_gscale": SVC(
            C=1.0,
            gamma="scale",
            kernel="rbf",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "rf_leaf2": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "extraTrees_leaf1": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


def build_pipeline(model, k, n_features):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
    if k != "all":
        actual_k = min(int(k), n_features)
        steps.append(("select", SelectKBest(f_classif, k=actual_k)))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def metric_dict(y_true, y_pred, label_order, prefix=""):
    out = {
        prefix + "accuracy": accuracy_score(y_true, y_pred),
        prefix + "macro_f1": f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0),
        prefix + "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }

    pr, rc, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=label_order, zero_division=0
    )

    for lab, p, r, f, s in zip(label_order, pr, rc, f1, sup):
        safe = safe_name(lab)
        out[prefix + f"precision_{safe}"] = p
        out[prefix + f"recall_{safe}"] = r
        out[prefix + f"f1_{safe}"] = f
        out[prefix + f"support_{safe}"] = int(s)

    return out


In [5]:
# ================================================================
# LOAD DATASETS AND CREATE TASKS
# ================================================================

# ---------- Load activity 10s dataset ----------
if not os.path.exists(ACTIVITY_DATA_PATH):
    raise FileNotFoundError(
        f"Could not find activity dataset:\n{ACTIVITY_DATA_PATH}\n"
        "Run the advanced merged 10s feature-generation cell first."
    )

activity_df_raw = pd.read_csv(ACTIVITY_DATA_PATH)
activity_df_raw["recognition_label"] = activity_df_raw["recognition_label"].astype(str).str.strip()
activity_df_raw = add_elapsed_min(activity_df_raw, "group", "window_start", "window_end")
activity_window_s = infer_window_seconds(activity_df_raw, "window_start", "window_end", default=10.0)

print("Loaded activity dataset:", ACTIVITY_DATA_PATH)
print("Shape:", activity_df_raw.shape)
print("Estimated window seconds:", activity_window_s)
display(activity_df_raw["recognition_label"].value_counts())

# ---------- Load binary interaction 5s dataset ----------
interaction_path = find_first_existing(INTERACTION_DATA_PATHS)
if interaction_path is None:
    print("WARNING: interaction binary dataset not found. Interaction task will be skipped.")
    interaction_df_raw = None
    interaction_window_s = 5.0
else:
    interaction_df_raw = pd.read_csv(interaction_path)
    interaction_df_raw["binary_label"] = interaction_df_raw["binary_label"].astype(str).str.strip()
    interaction_df_raw = add_elapsed_min(interaction_df_raw, "group", "window_start", "window_end")
    interaction_window_s = infer_window_seconds(interaction_df_raw, "window_start", "window_end", default=5.0)

    print("\nLoaded interaction dataset:", interaction_path)
    print("Shape:", interaction_df_raw.shape)
    print("Estimated window seconds:", interaction_window_s)
    display(interaction_df_raw["binary_label"].value_counts())


# ---------- Feature extraction per dataset ----------
def get_modality_features(df, label_cols=None):
    label_cols = label_cols or []

    oe_raw = [c for c in df.columns if is_oe_feature(c)]
    opti_raw = [c for c in df.columns if is_opti_feature(c)]
    xsens_raw = [c for c in df.columns if is_xsens_feature(c)]

    # Make modality sets mutually exclusive if broad fallback captured something unexpectedly.
    oe = clean_feature_list(df, oe_raw, label_cols=label_cols, include_elapsed=False)
    opti = clean_feature_list(df, [c for c in opti_raw if c not in oe], label_cols=label_cols, include_elapsed=False)
    xsens = clean_feature_list(df, [c for c in xsens_raw if c not in oe and c not in opti], label_cols=label_cols, include_elapsed=False)

    return {"OE": oe, "OPTI": opti, "XSENS": xsens}


def make_combo_feature_sets(df, modality_features, label_cols=None):
    combo_sets = {}
    combo_sets_elapsed = {}

    for combo_name, modalities in SENSOR_COMBINATIONS.items():
        feats = []
        for m in modalities:
            feats.extend(modality_features.get(m, []))
        feats = clean_feature_list(df, feats, label_cols=label_cols, include_elapsed=False)
        feats_elapsed = clean_feature_list(
            df,
            feats + (["elapsed_min"] if "elapsed_min" in df.columns else []),
            label_cols=label_cols,
            include_elapsed=True,
        )
        combo_sets[combo_name] = feats
        combo_sets_elapsed[combo_name] = feats_elapsed

    return combo_sets, combo_sets_elapsed


# ---------- Task preparation ----------
def prepare_task(task_name):
    if task_name == "interaction_vs_noninteraction":
        if interaction_df_raw is None:
            return None

        df = interaction_df_raw.copy()
        label_col = "binary_label"
        labels = ["interaction", "non_interaction"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [6, 12, 18]   # 5s windows -> 30/60/90s
        window_s = interaction_window_s

    elif task_name == "conversation_vs_nonconversation":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        core = ["co_building", "co_merging", "conversation"]
        df = df[df[label_col].isin(core)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = np.where(df[label_col] == "conversation", "conversation", "non_conversation")
        labels = ["conversation", "non_conversation"]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "conversation_vs_building":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["conversation", "co_building"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "conversation_vs_merging":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["conversation", "co_merging"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "merging_vs_building":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["co_merging", "co_building"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "three_class_activity":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["co_building", "co_merging", "conversation"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    else:
        raise ValueError(f"Unknown task: {task_name}")

    if len(df) == 0:
        print("Skipping empty task:", task_name)
        return None

    modality_features = get_modality_features(df, label_cols=[label_col, target_col])
    combo_sets, combo_sets_elapsed = make_combo_feature_sets(df, modality_features, label_cols=[label_col, target_col])

    spec = {
        "task_name": task_name,
        "df": df,
        "label_col": label_col,
        "target_col": target_col,
        "label_order": labels,
        "group_col": "group",
        "start_col": "window_start",
        "end_col": "window_end",
        "window_seconds": window_s,
        "seq_lens": default_seq_lens,
        "modality_features": modality_features,
        "combo_features": combo_sets,
        "combo_features_elapsed": combo_sets_elapsed,
        "out_dir": os.path.join(OUT_DIR, task_name),
    }
    os.makedirs(spec["out_dir"], exist_ok=True)
    return spec


task_specs = []
for task in RUN_TASKS:
    spec = prepare_task(task)
    if spec is not None:
        task_specs.append(spec)

print("\n" + "=" * 100)
print("PREPARED TASKS AND SENSOR COMBINATIONS")
print("=" * 100)

for spec in task_specs:
    print("\nTASK:", spec["task_name"])
    print("Rows:", len(spec["df"]))
    print("Classes:", spec["label_order"])
    print("Window seconds:", spec["window_seconds"])
    print("Seq lens:", spec["seq_lens"])
    print("Modality counts:", {k: len(v) for k, v in spec["modality_features"].items()})
    print("Combination counts, no elapsed:", {k: len(v) for k, v in spec["combo_features"].items()})
    print("Combination counts, with elapsed:", {k: len(v) for k, v in spec["combo_features_elapsed"].items()})
    display(spec["df"][spec["target_col"]].value_counts())
    display(pd.crosstab(spec["df"][spec["group_col"]], spec["df"][spec["target_col"]]))

    feature_count_rows = []
    for combo in SENSOR_COMBINATIONS:
        feature_count_rows.append({
            "task": spec["task_name"],
            "sensor_combo": combo,
            "n_no_elapsed": len(spec["combo_features"][combo]),
            "n_with_elapsed": len(spec["combo_features_elapsed"][combo]),
        })
    pd.DataFrame(feature_count_rows).to_csv(os.path.join(spec["out_dir"], f"{spec['task_name']}_feature_counts.csv"), index=False)


Loaded activity dataset: /content/drive/MyDrive/thesis/data/INTERACTION_ABLATIONS/activity3_advanced_merged_10s_features.csv
Shape: (992, 1543)
Estimated window seconds: 10.0


,count
recognition_label,
co_building,548
conversation,311
co_merging,133



Loaded interaction dataset: /content/drive/MyDrive/thesis/data/INTERACTION_BINARY_5S_SPECIALIZED_OE/binary_5s_specialized_oe_merged_all_features.csv
Shape: (4579, 1515)
Estimated window seconds: 5.0


,count
binary_label,
non_interaction,2304
interaction,2275



PREPARED TASKS AND SENSOR COMBINATIONS

TASK: conversation_vs_nonconversation
Rows: 992
Classes: ['conversation', 'non_conversation']
Window seconds: 10.0
Seq lens: [3, 6, 9]
Modality counts: {'OE': 305, 'OPTI': 480, 'XSENS': 692}
Combination counts, no elapsed: {'OE': 305, 'OPTI': 480, 'XSENS': 692, 'OE_OPTI': 785, 'OE_XSENS': 997, 'OPTI_XSENS': 1172, 'OE_OPTI_XSENS': 1477}
Combination counts, with elapsed: {'OE': 306, 'OPTI': 481, 'XSENS': 693, 'OE_OPTI': 786, 'OE_XSENS': 998, 'OPTI_XSENS': 1173, 'OE_OPTI_XSENS': 1478}


...

,count
task_label,
co_building,548
co_merging,133


task_label,co_building,co_merging
group,,
1,9,43
2,80,0
3,39,8
5,89,28
6,43,10
7,113,24
8,54,4
9,107,11
10,14,5



TASK: three_class_activity
Rows: 992
Classes: ['co_building', 'co_merging', 'conversation']
Window seconds: 10.0
Seq lens: [3, 6, 9]
Modality counts: {'OE': 305, 'OPTI': 480, 'XSENS': 692}
Combination counts, no elapsed: {'OE': 305, 'OPTI': 480, 'XSENS': 692, 'OE_OPTI': 785, 'OE_XSENS': 997, 'OPTI_XSENS': 1172, 'OE_OPTI_XSENS': 1477}
Combination counts, with elapsed: {'OE': 306, 'OPTI': 481, 'XSENS': 693, 'OE_OPTI': 786, 'OE_XSENS': 998, 'OPTI_XSENS': 1173, 'OE_OPTI_XSENS': 1478}


,count
task_label,
co_building,548
conversation,311
co_merging,133


task_label,co_building,co_merging,conversation
group,,,
1,9,43,57
2,80,0,26
3,39,8,42
5,89,28,13
6,43,10,29
7,113,24,25
8,54,4,49
9,107,11,54
10,14,5,16


## Safe execution and recovery

Run the notebook from top to bottom. The notebook now checks Google Drive before
starting each expensive section.

- If the complete classical CSV files already exist, the classical grid is
  reloaded automatically.
- Every completed DL task/sensor/model configuration is saved separately.
- After a runtime disconnect, rerun the mount, setup, configuration, data and
  helper cells, then rerun the DL cell. Completed configurations will display
  `already completed` and will be skipped.
- The final publication-table cell reloads both classical and DL outputs
  automatically when the Python variables were lost.

Do not delete the `PUBLICATION_TASK2_FULL_COMPARISON` output directory until the
full experiment is finished.

## Classical comparison

Default report-reproduction mode evaluates Logistic Regression and
Linear SVC with `k=80` and `k=200`, matching the complete comparison
table in the report.

Set `REPORT_REPRODUCTION_MODE=False` in the configuration cell to
additionally evaluate RBF-SVC, Random Forest and Extra Trees over the
larger exploratory k-grid.

In [6]:
# ================================================================
# CLASSICAL LOGO RUNNER WITH FOLD MEAN ± SD
# ================================================================

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut


CLASSICAL_SUMMARY_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_summary_with_std.csv",
)
CLASSICAL_FOLDS_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_fold_metrics.csv",
)
CLASSICAL_BEST_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_best_per_condition_with_std.csv",
)

CLASSICAL_FILES_COMPLETE = all(
    os.path.exists(path)
    for path in [
        CLASSICAL_SUMMARY_PATH,
        CLASSICAL_FOLDS_PATH,
        CLASSICAL_BEST_PATH,
    ]
)

if RESUME_EXISTING and CLASSICAL_FILES_COMPLETE:
    print("Completed classical results found. Reloading them from Drive.")
    combined_classical = pd.read_csv(CLASSICAL_SUMMARY_PATH)
    combined_classical_folds = pd.read_csv(CLASSICAL_FOLDS_PATH)
    combined_classical_best = pd.read_csv(CLASSICAL_BEST_PATH)

    # Prevent the long classical grid from running again in this session.
    RUN_CLASSICAL = False

    print("Classical summary rows:", len(combined_classical))
    print("Classical fold rows:", len(combined_classical_folds))
    print("Best-condition rows:", len(combined_classical_best))
    display(combined_classical_best.round(4))


if REPORT_REPRODUCTION_MODE:
    CLASSICAL_MODEL_NAMES = ["logreg_C1", "linearSVC_C1"]
    CLASSICAL_K_VALUES = [80, 200]
else:
    CLASSICAL_MODEL_NAMES = [
        "logreg_C1",
        "linearSVC_C1",
        "rbfSVC_C1_gscale",
        "rf_leaf2",
        "extraTrees_leaf1",
    ]
    CLASSICAL_K_VALUES = K_CLASSICAL_FULL


def run_classical_for_task_and_sensor(spec, sensor_combo):
    task_name = spec["task_name"]
    df = spec["df"]

    y = df[spec["target_col"]].astype(str).values
    groups = df[spec["group_col"]].values
    label_order = spec["label_order"]
    n_classes = len(label_order)

    all_models = make_classical_models(n_classes)
    models = {
        name: all_models[name]
        for name in CLASSICAL_MODEL_NAMES
        if name in all_models
    }

    summary_rows = []
    fold_rows = []
    pred_rows = []

    logo = LeaveOneGroupOut()

    for time_condition in TIME_CONDITIONS:
        if time_condition == "no_elapsed":
            feats = spec["combo_features"][sensor_combo]
        else:
            feats = spec["combo_features_elapsed"][sensor_combo]

        if len(feats) == 0:
            print(
                f"Skipping {task_name} | {sensor_combo} | "
                f"{time_condition}: no usable features"
            )
            continue

        X = (
            df[feats]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .values
        )

        for model_name, model in models.items():
            for k in CLASSICAL_K_VALUES:
                if k != "all" and int(k) > len(feats):
                    continue

                print(
                    f"Classical | {task_name} | {sensor_combo} | "
                    f"{time_condition} | {model_name} | k={k}"
                )

                pipe = build_pipeline(model, k, len(feats))

                y_true_all = []
                y_pred_all = []
                group_all = []
                row_index_all = []

                for fold, (tr_idx, te_idx) in enumerate(
                    logo.split(X, y, groups), start=1
                ):
                    if len(np.unique(y[tr_idx])) < 2:
                        continue

                    pipe_fold = clone(pipe)
                    pipe_fold.fit(X[tr_idx], y[tr_idx])
                    pred = pipe_fold.predict(X[te_idx])

                    fold_metric = metric_dict(
                        y[te_idx], pred, label_order
                    )
                    fold_rows.append({
                        "task": task_name,
                        "sensor_combo": sensor_combo,
                        "time_condition": time_condition,
                        "model": model_name,
                        "k": k,
                        "n_features": len(feats),
                        "fold": fold,
                        "test_group": groups[te_idx][0],
                        "n_test_rows": len(te_idx),
                        "accuracy": fold_metric["accuracy"],
                        "macro_f1": fold_metric["macro_f1"],
                        "balanced_accuracy": fold_metric[
                            "balanced_accuracy"
                        ],
                    })

                    y_true_all.extend(y[te_idx])
                    y_pred_all.extend(pred)
                    group_all.extend(groups[te_idx])
                    row_index_all.extend(te_idx)

                if not y_true_all:
                    continue

                y_true_all = np.asarray(y_true_all)
                y_pred_all = np.asarray(y_pred_all)

                pooled = metric_dict(
                    y_true_all, y_pred_all, label_order
                )
                pooled.update({
                    "task": task_name,
                    "sensor_combo": sensor_combo,
                    "time_condition": time_condition,
                    "model": model_name,
                    "k": k,
                    "n_features": len(feats),
                    "n_rows_evaluated": len(y_true_all),
                })
                summary_rows.append(pooled)

                if SAVE_CLASSICAL_PREDICTIONS:
                    for idx, g, yt, yp in zip(
                        row_index_all,
                        group_all,
                        y_true_all,
                        y_pred_all,
                    ):
                        pred_rows.append({
                            "task": task_name,
                            "sensor_combo": sensor_combo,
                            "time_condition": time_condition,
                            "model": model_name,
                            "k": k,
                            "row_index": int(idx),
                            "group": g,
                            "y_true": yt,
                            "y_pred": yp,
                            "correct": bool(yt == yp),
                        })

    summary = pd.DataFrame(summary_rows)
    folds = pd.DataFrame(fold_rows)
    preds = pd.DataFrame(pred_rows)

    if summary.empty:
        return summary, folds, preds, pd.DataFrame()

    group_keys = [
        "task",
        "sensor_combo",
        "time_condition",
        "model",
        "k",
        "n_features",
    ]

    fold_stats = (
        folds.groupby(group_keys, as_index=False)
        .agg(
            fold_accuracy_mean=("accuracy", "mean"),
            fold_accuracy_std=("accuracy", "std"),
            fold_macro_f1_mean=("macro_f1", "mean"),
            fold_macro_f1_std=("macro_f1", "std"),
            fold_balanced_accuracy_mean=(
                "balanced_accuracy", "mean"
            ),
            fold_balanced_accuracy_std=(
                "balanced_accuracy", "std"
            ),
            n_folds=("test_group", "nunique"),
        )
    )

    summary = summary.merge(
        fold_stats,
        on=group_keys,
        how="left",
        validate="one_to_one",
    )

    summary = (
        summary.sort_values(
            ["macro_f1", "accuracy"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    # Match the report: select the best configuration by pooled macro-F1.
    best_per_condition = (
        summary.sort_values(
            ["time_condition", "macro_f1", "accuracy"],
            ascending=[True, False, False],
        )
        .groupby("time_condition", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    out_dir = os.path.join(spec["out_dir"], sensor_combo)
    os.makedirs(out_dir, exist_ok=True)

    summary.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_summary_with_std.csv",
        ),
        index=False,
    )
    folds.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_fold_metrics.csv",
        ),
        index=False,
    )
    best_per_condition.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_best_per_condition_with_std.csv",
        ),
        index=False,
    )

    if SAVE_CLASSICAL_PREDICTIONS and not preds.empty:
        preds.to_csv(
            os.path.join(
                out_dir,
                f"{task_name}_{sensor_combo}_classical_predictions.csv",
            ),
            index=False,
        )

    display(best_per_condition.round(4))
    return summary, folds, preds, best_per_condition


all_classical_summaries = []
all_classical_folds = []
all_classical_best = []

if RUN_CLASSICAL:
    print("Classical models:", CLASSICAL_MODEL_NAMES)
    print("Classical k values:", CLASSICAL_K_VALUES)

    for spec in task_specs:
        for sensor_combo in SENSOR_COMBINATIONS:
            print("\n" + "=" * 110)
            print(
                "CLASSICAL:",
                spec["task_name"],
                "|",
                sensor_combo,
            )
            print("=" * 110)

            summary, folds, preds, best = (
                run_classical_for_task_and_sensor(
                    spec,
                    sensor_combo,
                )
            )

            if not summary.empty:
                all_classical_summaries.append(summary)
            if not folds.empty:
                all_classical_folds.append(folds)
            if not best.empty:
                all_classical_best.append(best)

    if all_classical_summaries:
        combined_classical = pd.concat(
            all_classical_summaries,
            ignore_index=True,
        )
        combined_classical.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_summary_with_std.csv",
            ),
            index=False,
        )

    if all_classical_folds:
        combined_classical_folds = pd.concat(
            all_classical_folds,
            ignore_index=True,
        )
        combined_classical_folds.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_fold_metrics.csv",
            ),
            index=False,
        )

    if all_classical_best:
        combined_classical_best = pd.concat(
            all_classical_best,
            ignore_index=True,
        )
        combined_classical_best.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_best_per_condition_with_std.csv",
            ),
            index=False,
        )
        display(combined_classical_best.round(4))


Classical models: ['logreg_C1', 'linearSVC_C1']
Classical k values: [80, 200]

CLASSICAL: conversation_vs_nonconversation | OE
Classical | conversation_vs_nonconversation | OE | no_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | OE | no_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | OE | no_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | OE | no_elapsed | linearSVC_C1 | k=200
Classical | conversation_vs_nonconversation | OE | with_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | OE | with_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | OE | with_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | OE | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_conversation,recall_conversation,f1_conversation,support_conversation,precision_non_conversation,recall_non_conversation,f1_non_conversation,support_non_conversation,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.7177,0.6885,0.7010,0.5411,0.6559,0.5930,311,0.8260,0.7460,0.7840,681,conversation_vs_nonconversation,OE,no_elapsed,logreg_C1,80,305,992,0.7084,0.1606,0.6597,0.1503,0.6693,0.1183,9
1,0.8075,0.7810,0.7873,0.6786,0.7331,0.7048,311,0.8735,0.8414,0.8571,681,conversation_vs_nonconversation,OE,with_elapsed,logreg_C1,80,306,992,0.7654,0.1922,0.6900,0.2140,0.7218,0.1433,9



CLASSICAL: conversation_vs_nonconversation | OPTI
Classical | conversation_vs_nonconversation | OPTI | no_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | OPTI | no_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | OPTI | no_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | OPTI | no_elapsed | linearSVC_C1 | k=200
Classical | conversation_vs_nonconversation | OPTI | with_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | OPTI | with_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | OPTI | with_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | OPTI | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_conversation,recall_conversation,f1_conversation,support_conversation,precision_non_conversation,recall_non_conversation,f1_non_conversation,support_non_conversation,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.8417,0.8194,0.8253,0.7319,0.7814,0.7558,311,0.8970,0.8693,0.8829,681,conversation_vs_nonconversation,OPTI,no_elapsed,logreg_C1,80,480,992,0.8442,0.0568,0.8016,0.0878,0.8160,0.0627,9
1,0.8609,0.8433,0.8550,0.7479,0.8392,0.7909,311,0.9222,0.8708,0.8958,681,conversation_vs_nonconversation,OPTI,with_elapsed,linearSVC_C1,80,481,992,0.8324,0.1295,0.7887,0.1477,0.8064,0.1030,9



CLASSICAL: conversation_vs_nonconversation | XSENS
Classical | conversation_vs_nonconversation | XSENS | no_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | XSENS | no_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | conversation_vs_nonconversation | XSENS | with_elapsed | logreg_C1 | k=80
Classical | conversation_vs_nonconversation | XSENS | with_elapsed | logreg_C1 | k=200
Classical | conversation_vs_nonconversation | XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | conversation_vs_nonconversation | XSENS | with_elapsed | linearSVC_C1 | k=200


...


CLASSICAL: three_class_activity | OPTI_XSENS
Classical | three_class_activity | OPTI_XSENS | no_elapsed | logreg_C1 | k=80
Classical | three_class_activity | OPTI_XSENS | no_elapsed | logreg_C1 | k=200
Classical | three_class_activity | OPTI_XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | three_class_activity | OPTI_XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | three_class_activity | OPTI_XSENS | with_elapsed | logreg_C1 | k=80
Classical | three_class_activity | OPTI_XSENS | with_elapsed | logreg_C1 | k=200
Classical | three_class_activity | OPTI_XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | three_class_activity | OPTI_XSENS | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_co_building,recall_co_building,f1_co_building,support_co_building,precision_co_merging,recall_co_merging,f1_co_merging,support_co_merging,precision_conversation,recall_conversation,f1_conversation,support_conversation,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.6946,0.6531,0.6879,0.8225,0.6679,0.7372,548,0.3722,0.6241,0.4663,133,0.7407,0.7717,0.7559,311,three_class_activity,OPTI_XSENS,no_elapsed,logreg_C1,80,1172,992,0.7133,0.1020,0.6025,0.1235,0.6643,0.1247,9
1,0.7268,0.6284,0.6303,0.7926,0.7810,0.7868,548,0.3362,0.2932,0.3133,133,0.7560,0.8167,0.7852,311,three_class_activity,OPTI_XSENS,with_elapsed,linearSVC_C1,80,1173,992,0.7208,0.1019,0.5579,0.1124,0.6023,0.1274,9



CLASSICAL: three_class_activity | OE_OPTI_XSENS
Classical | three_class_activity | OE_OPTI_XSENS | no_elapsed | logreg_C1 | k=80
Classical | three_class_activity | OE_OPTI_XSENS | no_elapsed | logreg_C1 | k=200
Classical | three_class_activity | OE_OPTI_XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | three_class_activity | OE_OPTI_XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | three_class_activity | OE_OPTI_XSENS | with_elapsed | logreg_C1 | k=80
Classical | three_class_activity | OE_OPTI_XSENS | with_elapsed | logreg_C1 | k=200
Classical | three_class_activity | OE_OPTI_XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | three_class_activity | OE_OPTI_XSENS | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_co_building,recall_co_building,f1_co_building,support_co_building,precision_co_merging,recall_co_merging,f1_co_merging,support_co_merging,precision_conversation,recall_conversation,f1_conversation,support_conversation,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.6946,0.6530,0.6874,0.8210,0.6697,0.7377,548,0.3705,0.6241,0.4650,133,0.7445,0.7685,0.7563,311,three_class_activity,OE_OPTI_XSENS,no_elapsed,logreg_C1,80,1477,992,0.7133,0.1020,0.6017,0.1215,0.6639,0.1238,9
1,0.7218,0.6234,0.6254,0.7837,0.7737,0.7787,548,0.3276,0.2857,0.3052,133,0.7582,0.8167,0.7864,311,three_class_activity,OE_OPTI_XSENS,with_elapsed,linearSVC_C1,80,1478,992,0.7140,0.0985,0.5495,0.1073,0.5951,0.1254,9


,accuracy,macro_f1,balanced_accuracy,precision_conversation,recall_conversation,f1_conversation,support_conversation,precision_non_conversation,recall_non_conversation,f1_non_conversation,support_non_conversation,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds,precision_co_building,recall_co_building,f1_co_building,support_co_building,precision_co_merging,recall_co_merging,f1_co_merging,support_co_merging
0,0.7177,0.6885,0.7010,0.5411,0.6559,0.5930,311.0,0.8260,0.7460,0.7840,681.0,conversation_vs_nonconversation,OE,no_elapsed,logreg_C1,80,305,992,0.7084,0.1606,0.6597,0.1503,0.6693,0.1183,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.8075,0.7810,0.7873,0.6786,0.7331,0.7048,311.0,0.8735,0.8414,0.8571,681.0,conversation_vs_nonconversation,OE,with_elapsed,logreg_C1,80,306,992,0.7654,0.1922,0.6900,0.2140,0.7218,0.1433,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.8417,0.8194,0.8253,0.7319,0.7814,0.7558,311.0,0.8970,0.8693,0.8829,681.0,conversation_vs_nonconversation,OPTI,no_elapsed,logreg_C1,80,480,992,0.8442,0.0568,0.8016,0.0878,0.8160,0.0627,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.8609,0.8433,0.8550,0.7479,0.8392,0.7909,311.0,0.9222,0.8708,0.8958,681.0,conversation_vs_nonconversation,OPTI,with_elapsed,linearSVC_C1,80,481,992,0.8324,0.1295,0.7887,0.1477,0.8064,0.1030,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.6724,0.6285,0.6321,0.4794,0.5241,0.5008,311.0,0.7730,0.7401,0.7562,681.0,conversation_vs_nonconversation,XSENS,no_elapsed,logreg_C1,80,692,992,0.6414,0.1152,0.5680,0.1330,0.5835,0.1091,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.8125,0.7901,0.8023,0.6751,0.7749,0.7216,311.0,0.8898,0.8297,0.8587,681.0,conversation_vs_nonconversation,XSENS,with_elapsed,logreg_C1,80,693,992,0.7722,0.1651,0.7121,0.1903,0.7507,0.1196,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.8306,0.8078,0.8155,0.7109,0.7749,0.7415,311.0,0.8928,0.8561,0.8741,681.0,conversation_vs_nonconversation,OE_OPTI,no_elapsed,linearSVC_C1,80,785,992,0.8295,0.0559,0.7842,0.0814,0.8002,0.0611,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.8407,0.8190,0.8263,0.7270,0.7878,0.7562,311.0,0.8992,0.8649,0.8817,681.0,conversation_vs_nonconversation,OE_OPTI,with_elapsed,logreg_C1,80,786,992,0.8150,0.1250,0.7656,0.1383,0.7802,0.0952,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.7016,0.6724,0.6857,0.5195,0.6431,0.5747,311.0,0.8171,0.7283,0.7702,681.0,conversation_vs_nonconversation,OE_XSENS,no_elapsed,logreg_C1,200,997,992,0.6948,0.1778,0.6563,0.1651,0.6848,0.1100,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.7792,0.7519,0.7615,0.6307,0.7138,0.6697,311.0,0.8609,0.8091,0.8342,681.0,conversation_vs_nonconversation,OE_XSENS,with_elapsed,logreg_C1,200,998,992,0.7508,0.1674,0.6885,0.1909,0.7293,0.1348,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Deep-learning comparison without elapsed time

LSTM, BiLSTM, GRU and Transformer are evaluated using the longest
context used in the original comparison: 90 seconds. Model selection
is performed separately for every task and sensor combination.

In [7]:
# ================================================================
# DL MODEL HELPERS
# ================================================================

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, rnn_type="lstm", hidden_dim=64, num_layers=1, dropout=0.25, bidirectional=False):
        super().__init__()
        rnn_cls = nn.LSTM if rnn_type == "lstm" else nn.GRU
        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, n_classes),
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.head(last)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.25):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model=d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        out = self.encoder(x)
        last = out[:, -1, :]
        return self.head(last)


ALL_MODEL_CONFIGS = [
    {"model_type": "lstm", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "bilstm", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "gru", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "transformer", "d_model": 64, "nhead": 4, "num_layers": 2, "dim_feedforward": 128, "dropout": 0.25, "lr": 5e-4, "weight_decay": 1e-4},
]

MODEL_CONFIGS = [m for m in ALL_MODEL_CONFIGS if m["model_type"] in RUN_DL_MODEL_TYPES]
print("DL model types:", [m["model_type"] for m in MODEL_CONFIGS])


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model(model_cfg, input_dim, n_classes):
    mt = model_cfg["model_type"]
    if mt == "lstm":
        return RNNClassifier(input_dim, n_classes, rnn_type="lstm", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=False)
    if mt == "bilstm":
        return RNNClassifier(input_dim, n_classes, rnn_type="lstm", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=True)
    if mt == "gru":
        return RNNClassifier(input_dim, n_classes, rnn_type="gru", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=False)
    if mt == "transformer":
        return TransformerClassifier(input_dim, n_classes, d_model=model_cfg["d_model"], nhead=model_cfg["nhead"], num_layers=model_cfg["num_layers"], dim_feedforward=model_cfg["dim_feedforward"], dropout=model_cfg["dropout"])
    raise ValueError(mt)


def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def make_sequences(X, y, groups, starts, seq_len):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y)
    groups = np.asarray(groups)
    starts = np.asarray(starts, dtype=float)

    Xs, ys, gs, sts = [], [], [], []

    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        idx = idx[np.argsort(starts[idx])]
        if len(idx) < seq_len:
            continue
        for end_pos in range(seq_len - 1, len(idx)):
            win_idx = idx[end_pos - seq_len + 1:end_pos + 1]
            Xs.append(X[win_idx])
            ys.append(y[idx[end_pos]])
            gs.append(g)
            sts.append(starts[idx[end_pos]])

    if len(Xs) == 0:
        return np.empty((0, seq_len, X.shape[1]), dtype=np.float32), np.array([]), np.array([]), np.array([])

    return np.stack(Xs).astype(np.float32), np.array(ys), np.array(gs), np.array(sts)


def torch_predict(model, X, batch_size=512):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            logits = model(xb)
            preds.extend(logits.argmax(1).cpu().numpy())
    return np.array(preds)


def choose_validation_group(train_groups, y_all, groups_all):
    candidates = []
    for g in sorted(np.unique(train_groups)):
        mask = groups_all == g
        n_classes = len(np.unique(y_all[mask]))
        n_rows = int(mask.sum())
        candidates.append((n_classes >= 2, n_rows, g))
    candidates = sorted(candidates, reverse=True)
    return candidates[0][2]


DL model types: ['lstm', 'bilstm', 'gru', 'transformer']


In [8]:
# ================================================================
# DL NO-ELAPSED LOGO RUNNER WITH FOLD MEAN ± SD
# ================================================================

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif

FAST_DL_TASKS_TO_RUN = None
FAST_DL_SENSOR_COMBOS_TO_RUN = None
FAST_DL_MODEL_TYPES = RUN_DL_MODEL_TYPES
FAST_DL_K = 120
FAST_DL_SEQ_CHOICE = "last"

FAST_DL_OUT_DIR = os.path.join(
    OUT_DIR,
    "DL_NO_ELAPSED_ALL_ARCHITECTURES",
)
os.makedirs(FAST_DL_OUT_DIR, exist_ok=True)

FAST_DL_RUN_DIR = os.path.join(
    FAST_DL_OUT_DIR,
    "PER_CONFIGURATION_CHECKPOINTS",
)
os.makedirs(FAST_DL_RUN_DIR, exist_ok=True)


def choose_fast_seq_len(spec):
    seq_lens = list(spec["seq_lens"])
    if FAST_DL_SEQ_CHOICE == "last":
        return seq_lens[-1]
    if FAST_DL_SEQ_CHOICE == "first":
        return seq_lens[0]
    if FAST_DL_SEQ_CHOICE == "middle":
        return seq_lens[len(seq_lens) // 2]
    return int(FAST_DL_SEQ_CHOICE)


def get_fast_model_configs():
    return [
        cfg
        for cfg in MODEL_CONFIGS
        if cfg["model_type"] in FAST_DL_MODEL_TYPES
    ]


def torch_predict_fast(
    model,
    X,
    batch_size=FAST_PRED_BATCH_SIZE,
):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(
                X[i:i + batch_size],
                dtype=torch.float32,
            ).to(DEVICE)
            preds.extend(
                model(xb).argmax(1).cpu().numpy()
            )

    return np.asarray(preds)


def train_one_fast_dl_run(
    spec,
    sensor_combo,
    seq_len,
    k_features,
    model_cfg,
    seed,
):
    set_seed(seed)

    task_name = spec["task_name"]
    df = spec["df"]
    feature_list = spec["combo_features"][
        sensor_combo
    ].copy()
    label_order = spec["label_order"]
    n_classes = len(label_order)

    if not feature_list:
        return None, None, None

    label_to_id = {
        label: index
        for index, label in enumerate(label_order)
    }
    id_to_label = {
        index: label
        for label, index in label_to_id.items()
    }

    y_label = df[spec["target_col"]].astype(str).values
    y_all = np.asarray(
        [label_to_id[value] for value in y_label],
        dtype=int,
    )

    groups_all = df[spec["group_col"]].values
    starts_all = pd.to_numeric(
        df[spec["start_col"]],
        errors="coerce",
    ).values

    X_raw = (
        df[feature_list]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .values
    )

    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    fold_rows = []
    pred_rows = []

    for fold, (trval_idx, te_idx) in enumerate(
        logo.split(X_raw, y_all, groups_all),
        start=1,
    ):
        if (
            MAX_LOGO_FOLDS is not None
            and fold > MAX_LOGO_FOLDS
        ):
            break

        if len(np.unique(y_all[trval_idx])) < 2:
            continue

        test_group = groups_all[te_idx][0]

        train_groups = np.unique(groups_all[trval_idx])
        val_group = choose_validation_group(
            train_groups,
            y_all,
            groups_all,
        )

        val_mask = groups_all[trval_idx] == val_group
        val_idx = trval_idx[val_mask]
        tr_idx = trval_idx[~val_mask]

        if len(np.unique(y_all[tr_idx])) < 2:
            continue

        imputer = SimpleImputer(strategy="median")
        scaler = RobustScaler()

        Xtr = imputer.fit_transform(X_raw[tr_idx])
        Xval = imputer.transform(X_raw[val_idx])
        Xte = imputer.transform(X_raw[te_idx])

        Xtr = scaler.fit_transform(Xtr)
        Xval = scaler.transform(Xval)
        Xte = scaler.transform(Xte)

        actual_k = min(
            int(k_features),
            Xtr.shape[1],
        )

        selector = SelectKBest(
            f_classif,
            k=actual_k,
        )
        Xtr = selector.fit_transform(
            Xtr,
            y_all[tr_idx],
        )
        Xval = selector.transform(Xval)
        Xte = selector.transform(Xte)

        Xtr_seq, ytr_seq, _, _ = make_sequences(
            Xtr,
            y_all[tr_idx],
            groups_all[tr_idx],
            starts_all[tr_idx],
            seq_len,
        )
        Xval_seq, yval_seq, _, _ = make_sequences(
            Xval,
            y_all[val_idx],
            groups_all[val_idx],
            starts_all[val_idx],
            seq_len,
        )
        Xte_seq, yte_seq, gte_seq, ste_seq = (
            make_sequences(
                Xte,
                y_all[te_idx],
                groups_all[te_idx],
                starts_all[te_idx],
                seq_len,
            )
        )

        if (
            len(Xtr_seq) == 0
            or len(Xval_seq) == 0
            or len(Xte_seq) == 0
        ):
            continue

        model = build_model(
            model_cfg,
            input_dim=actual_k,
            n_classes=n_classes,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=model_cfg["lr"],
            weight_decay=model_cfg["weight_decay"],
        )
        loss_fn = nn.CrossEntropyLoss()

        train_loader = make_loader(
            Xtr_seq,
            ytr_seq,
            batch_size=FAST_BATCH_SIZE,
            shuffle=True,
        )

        best_state = None
        best_val_macro = -np.inf
        patience_left = FAST_PATIENCE
        best_epoch = 0

        for epoch in range(
            1,
            FAST_MAX_EPOCHS + 1,
        ):
            model.train()

            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0,
                )
                optimizer.step()

            val_pred = torch_predict_fast(
                model,
                Xval_seq,
            )
            val_macro = f1_score(
                yval_seq,
                val_pred,
                labels=np.arange(n_classes),
                average="macro",
                zero_division=0,
            )

            if val_macro > best_val_macro:
                best_val_macro = val_macro
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value
                    in model.state_dict().items()
                }
                patience_left = FAST_PATIENCE
                best_epoch = epoch
            else:
                patience_left -= 1

            if patience_left <= 0:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
            model.to(DEVICE)

        test_pred = torch_predict_fast(
            model,
            Xte_seq,
        )

        y_true_all.extend(yte_seq.tolist())
        y_pred_all.extend(test_pred.tolist())

        fold_metric = metric_dict(
            yte_seq,
            test_pred,
            list(range(n_classes)),
        )

        fold_rows.append({
            "task": task_name,
            "sensor_combo": sensor_combo,
            "time_condition": "no_elapsed",
            "model_type": model_cfg["model_type"],
            "seed": seed,
            "seq_len": seq_len,
            "context_seconds": (
                seq_len * spec["window_seconds"]
            ),
            "k_features": actual_k,
            "n_features_before_select": len(
                feature_list
            ),
            "fold": fold,
            "test_group": test_group,
            "n_sequences": len(Xte_seq),
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro,
            "accuracy": fold_metric["accuracy"],
            "macro_f1": fold_metric["macro_f1"],
            "balanced_accuracy": fold_metric[
                "balanced_accuracy"
            ],
        })

        if SAVE_FAST_DL_PREDICTIONS:
            for yt, yp, group, start in zip(
                yte_seq,
                test_pred,
                gte_seq,
                ste_seq,
            ):
                pred_rows.append({
                    "task": task_name,
                    "sensor_combo": sensor_combo,
                    "model_type": model_cfg["model_type"],
                    "seed": seed,
                    "seq_len": seq_len,
                    "group": group,
                    "window_start": start,
                    "y_true": id_to_label[int(yt)],
                    "y_pred": id_to_label[int(yp)],
                    "correct": int(yt) == int(yp),
                })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not y_true_all:
        return None, None, None

    y_true_labels = np.asarray([
        id_to_label[int(value)]
        for value in y_true_all
    ])
    y_pred_labels = np.asarray([
        id_to_label[int(value)]
        for value in y_pred_all
    ])

    summary = metric_dict(
        y_true_labels,
        y_pred_labels,
        label_order,
    )
    summary.update({
        "task": task_name,
        "sensor_combo": sensor_combo,
        "time_condition": "no_elapsed",
        "model_type": model_cfg["model_type"],
        "seed": seed,
        "seq_len": seq_len,
        "context_seconds": (
            seq_len * spec["window_seconds"]
        ),
        "k_features": int(k_features),
        "n_features_before_select": len(
            feature_list
        ),
        "n_sequences_evaluated": len(
            y_true_labels
        ),
        "max_epochs": FAST_MAX_EPOCHS,
        "patience": FAST_PATIENCE,
    })

    return (
        summary,
        pd.DataFrame(fold_rows),
        pd.DataFrame(pred_rows),
    )


fast_model_configs = get_fast_model_configs()

planned = []
for spec in task_specs:
    seq_len = choose_fast_seq_len(spec)

    for sensor_combo in SENSOR_COMBINATIONS:
        if not spec["combo_features"][sensor_combo]:
            continue

        for seed in SEEDS:
            for model_cfg in fast_model_configs:
                planned.append({
                    "task": spec["task_name"],
                    "sensor_combo": sensor_combo,
                    "model": model_cfg["model_type"],
                    "seed": seed,
                    "seq_len": seq_len,
                    "k": min(
                        FAST_DL_K,
                        len(
                            spec["combo_features"][
                                sensor_combo
                            ]
                        ),
                    ),
                })

planned_df = pd.DataFrame(planned)
display(planned_df)
planned_df.to_csv(
    os.path.join(
        FAST_DL_OUT_DIR,
        "dl_plan.csv",
    ),
    index=False,
)

all_fast_summaries = []
all_fast_folds = []
all_fast_preds = []

if RUN_DL:
    for run_index, row in planned_df.iterrows():
        spec = next(
            item
            for item in task_specs
            if item["task_name"] == row["task"]
        )
        model_cfg = next(
            item
            for item in fast_model_configs
            if item["model_type"] == row["model"]
        )

        run_key = (
            f"{row['task']}__{row['sensor_combo']}__"
            f"{row['model']}__seed{int(row['seed'])}__"
            f"seq{int(row['seq_len'])}__k{int(row['k'])}"
        )
        run_summary_path = os.path.join(
            FAST_DL_RUN_DIR,
            f"{run_key}__summary.csv",
        )
        run_folds_path = os.path.join(
            FAST_DL_RUN_DIR,
            f"{run_key}__folds.csv",
        )
        run_predictions_path = os.path.join(
            FAST_DL_RUN_DIR,
            f"{run_key}__predictions.csv",
        )

        if (
            RESUME_EXISTING
            and os.path.exists(run_summary_path)
            and os.path.exists(run_folds_path)
        ):
            print(
                f"DL {run_index + 1}/{len(planned_df)} already completed: "
                f"{run_key}. Reloading checkpoint."
            )

            saved_summary_df = pd.read_csv(run_summary_path)
            summary = saved_summary_df.iloc[0].to_dict()
            fold_df = pd.read_csv(run_folds_path)

            if (
                SAVE_FAST_DL_PREDICTIONS
                and os.path.exists(run_predictions_path)
            ):
                pred_df = pd.read_csv(run_predictions_path)
            else:
                pred_df = pd.DataFrame()

            all_fast_summaries.append(summary)

            if not fold_df.empty:
                all_fast_folds.append(fold_df)

            if (
                SAVE_FAST_DL_PREDICTIONS
                and not pred_df.empty
            ):
                all_fast_preds.append(pred_df)

            continue

        print("\n" + "=" * 110)
        print(
            f"DL {run_index + 1}/{len(planned_df)} | "
            f"{row['task']} | {row['sensor_combo']} | "
            f"{row['model']} | seed={row['seed']}"
        )
        print("=" * 110)

        summary, fold_df, pred_df = (
            train_one_fast_dl_run(
                spec=spec,
                sensor_combo=row["sensor_combo"],
                seq_len=int(row["seq_len"]),
                k_features=int(row["k"]),
                model_cfg=model_cfg,
                seed=int(row["seed"]),
            )
        )

        if summary is None:
            continue

        all_fast_summaries.append(summary)

        if not fold_df.empty:
            all_fast_folds.append(fold_df)

        if (
            SAVE_FAST_DL_PREDICTIONS
            and not pred_df.empty
        ):
            all_fast_preds.append(pred_df)

        # Save this complete task/sensor/model configuration immediately.
        # A later runtime disconnect will not require rerunning it.
        pd.DataFrame([summary]).to_csv(
            run_summary_path,
            index=False,
        )
        fold_df.to_csv(
            run_folds_path,
            index=False,
        )
        if (
            SAVE_FAST_DL_PREDICTIONS
            and not pred_df.empty
        ):
            pred_df.to_csv(
                run_predictions_path,
                index=False,
            )

        pd.DataFrame(
            all_fast_summaries
        ).to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "combined_dl_no_elapsed_summary_partial.csv",
            ),
            index=False,
        )

        if all_fast_folds:
            pd.concat(
                all_fast_folds,
                ignore_index=True,
            ).to_csv(
                os.path.join(
                    FAST_DL_OUT_DIR,
                    "combined_dl_no_elapsed_folds_partial.csv",
                ),
                index=False,
            )

if all_fast_summaries:
    combined_fast_dl = pd.DataFrame(
        all_fast_summaries
    )
    combined_fast_folds = pd.concat(
        all_fast_folds,
        ignore_index=True,
    )

    dl_group_keys = [
        "task",
        "sensor_combo",
        "time_condition",
        "model_type",
        "seed",
        "seq_len",
        "context_seconds",
        "k_features",
        "n_features_before_select",
    ]

    dl_fold_stats = (
        combined_fast_folds
        .groupby(
            dl_group_keys,
            as_index=False,
        )
        .agg(
            fold_accuracy_mean=("accuracy", "mean"),
            fold_accuracy_std=("accuracy", "std"),
            fold_macro_f1_mean=("macro_f1", "mean"),
            fold_macro_f1_std=("macro_f1", "std"),
            fold_balanced_accuracy_mean=(
                "balanced_accuracy", "mean"
            ),
            fold_balanced_accuracy_std=(
                "balanced_accuracy", "std"
            ),
            n_folds=("test_group", "nunique"),
        )
    )

    combined_fast_dl = combined_fast_dl.merge(
        dl_fold_stats,
        on=dl_group_keys,
        how="left",
        validate="one_to_one",
    )

    combined_fast_dl.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_summary_with_std.csv",
        ),
        index=False,
    )
    combined_fast_folds.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_fold_metrics.csv",
        ),
        index=False,
    )

    # Match the report: choose best by pooled macro-F1.
    combined_fast_dl_best = (
        combined_fast_dl
        .sort_values(
            [
                "task",
                "sensor_combo",
                "macro_f1",
                "accuracy",
            ],
            ascending=[
                True,
                True,
                False,
                False,
            ],
        )
        .groupby(
            ["task", "sensor_combo"],
            as_index=False,
        )
        .head(1)
        .reset_index(drop=True)
    )

    combined_fast_dl_best.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_best_per_task_sensor_with_std.csv",
        ),
        index=False,
    )

    display(combined_fast_dl_best.round(4))

    if len(SEEDS) > 1:
        seed_keys = [
            "task",
            "sensor_combo",
            "model_type",
            "seq_len",
            "context_seconds",
            "k_features",
            "n_features_before_select",
        ]
        dl_seed_stats = (
            combined_fast_dl
            .groupby(seed_keys, as_index=False)
            .agg(
                seed_accuracy_mean=("accuracy", "mean"),
                seed_accuracy_std=("accuracy", "std"),
                seed_macro_f1_mean=("macro_f1", "mean"),
                seed_macro_f1_std=("macro_f1", "std"),
                n_seeds=("seed", "nunique"),
            )
        )
        dl_seed_stats.to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "dl_seed_variability.csv",
            ),
            index=False,
        )
        display(dl_seed_stats.round(4))

    if (
        SAVE_FAST_DL_PREDICTIONS
        and all_fast_preds
    ):
        pd.concat(
            all_fast_preds,
            ignore_index=True,
        ).to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "combined_dl_no_elapsed_predictions.csv",
            ),
            index=False,
        )


,task,sensor_combo,model,seed,seq_len,k
0,conversation_vs_nonconversation,OE,lstm,42,9,120
1,conversation_vs_nonconversation,OE,bilstm,42,9,120
2,conversation_vs_nonconversation,OE,gru,42,9,120
3,conversation_vs_nonconversation,OE,transformer,42,9,120
4,conversation_vs_nonconversation,OPTI,lstm,42,9,120
5,conversation_vs_nonconversation,OPTI,bilstm,42,9,120
6,conversation_vs_nonconversation,OPTI,gru,42,9,120
7,conversation_vs_nonconversation,OPTI,transformer,42,9,120
8,conversation_vs_nonconversation,XSENS,lstm,42,9,120
9,conversation_vs_nonconversation,XSENS,bilstm,42,9,120



DL 1/140 | conversation_vs_nonconversation | OE | lstm | seed=42

DL 2/140 | conversation_vs_nonconversation | OE | bilstm | seed=42

DL 3/140 | conversation_vs_nonconversation | OE | gru | seed=42

DL 4/140 | conversation_vs_nonconversation | OE | transformer | seed=42

DL 5/140 | conversation_vs_nonconversation | OPTI | lstm | seed=42

DL 6/140 | conversation_vs_nonconversation | OPTI | bilstm | seed=42

DL 7/140 | conversation_vs_nonconversation | OPTI | gru | seed=42

DL 8/140 | conversation_vs_nonconversation | OPTI | transformer | seed=42

DL 9/140 | conversation_vs_nonconversation | XSENS | lstm | seed=42

DL 10/140 | conversation_vs_nonconversation | XSENS | bilstm | seed=42

DL 11/140 | conversation_vs_nonconversation | XSENS | gru | seed=42

DL 12/140 | conversation_vs_nonconversation | XSENS | transformer | seed=42

DL 13/140 | conversation_vs_nonconversation | OE_OPTI | lstm | seed=42

DL 14/140 | conversation_vs_nonconversation | OE_OPTI | bilstm | seed=42

DL 15/140 | co

,accuracy,macro_f1,balanced_accuracy,precision_conversation,recall_conversation,f1_conversation,support_conversation,precision_non_conversation,recall_non_conversation,f1_non_conversation,support_non_conversation,task,sensor_combo,time_condition,model_type,seed,seq_len,context_seconds,k_features,n_features_before_select,n_sequences_evaluated,max_epochs,patience,precision_co_building,recall_co_building,f1_co_building,support_co_building,precision_co_merging,recall_co_merging,f1_co_merging,support_co_merging,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.7560,0.6972,0.6889,0.6200,0.5167,0.5636,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OE,no_elapsed,transformer,42,9,90.0,120,305,787,25,4,0.8024,0.8611,0.8307,547.0,NaN,NaN,NaN,NaN,0.7253,0.1766,0.6109,0.1777,0.6570,0.1268,9
1,0.8806,0.8591,0.8591,0.8042,0.8042,0.8042,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OE_OPTI,no_elapsed,bilstm,42,9,90.0,120,785,787,25,4,0.9141,0.9141,0.9141,547.0,NaN,NaN,NaN,NaN,0.8698,0.0699,0.7727,0.1409,0.7735,0.1265,9
2,0.8806,0.8549,0.8451,0.8380,0.7542,0.7939,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OE_OPTI_XSENS,no_elapsed,transformer,42,9,90.0,120,1477,787,25,4,0.8967,0.9360,0.9159,547.0,NaN,NaN,NaN,NaN,0.8682,0.1013,0.7196,0.2217,0.7357,0.1914,9
3,0.7192,0.6480,0.6413,0.5492,0.4417,0.4896,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OE_XSENS,no_elapsed,transformer,42,9,90.0,120,997,787,25,4,0.7744,0.8410,0.8063,547.0,NaN,NaN,NaN,NaN,0.6639,0.2785,0.5181,0.2296,0.5926,0.1266,9
4,0.8755,0.8521,0.8496,0.8034,0.7833,0.7932,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OPTI,no_elapsed,lstm,42,9,90.0,120,480,787,25,4,0.9060,0.9159,0.9109,547.0,NaN,NaN,NaN,NaN,0.8607,0.1004,0.7387,0.2048,0.7584,0.1735,9
5,0.8806,0.8601,0.8626,0.7967,0.8167,0.8066,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,OPTI_XSENS,no_elapsed,bilstm,42,9,90.0,120,1172,787,25,4,0.9187,0.9086,0.9136,547.0,NaN,NaN,NaN,NaN,0.8676,0.1032,0.7482,0.2053,0.7658,0.1759,9
6,0.6518,0.6152,0.6268,0.4441,0.5625,0.4963,240.0,NaN,NaN,NaN,NaN,conversation_vs_building,XSENS,no_elapsed,lstm,42,9,90.0,120,692,787,25,4,0.7826,0.6910,0.7340,547.0,NaN,NaN,NaN,NaN,0.6280,0.1243,0.5341,0.1402,0.5779,0.1088,9
7,0.6640,0.6575,0.6801,0.8098,0.6234,0.7045,239.0,NaN,NaN,NaN,NaN,conversation_vs_merging,OE,no_elapsed,bilstm,42,9,90.0,120,305,372,25,4,NaN,NaN,NaN,NaN,0.5213,0.7368,0.6106,133.0,0.7073,0.1868,0.6303,0.1944,0.7179,0.1338,9
8,0.8790,0.8658,0.8592,0.8880,0.9289,0.9080,239.0,NaN,NaN,NaN,NaN,conversation_vs_merging,OE_OPTI,no_elapsed,transformer,42,9,90.0,120,785,372,25,4,NaN,NaN,NaN,NaN,0.8607,0.7895,0.8235,133.0,0.8978,0.0631,0.7755,0.1904,0.8517,0.1526,9
9,0.8737,0.8608,0.8567,0.8902,0.9163,0.9031,239.0,NaN,NaN,NaN,NaN,conversation_vs_merging,OE_OPTI_XSENS,no_elapsed,transformer,42,9,90.0,120,1477,372,25,4,NaN,NaN,NaN,NaN,0.8413,0.7970,0.8185,133.0,0.8959,0.0900,0.8275,0.1781,0.8900,0.0926,9


## Final publication tables

The long table keeps all numeric columns. The wide table mirrors the
structure of the report, with one column for each of the three regimes.

In [9]:
# ================================================================
# FINAL THREE-REGIME COMPARISON TABLES
# ================================================================


# ================================================================
# AUTOMATIC RECOVERY AFTER A COLAB RUNTIME DISCONNECT
# ================================================================

CLASSICAL_SUMMARY_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_summary_with_std.csv",
)
CLASSICAL_FOLDS_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_fold_metrics.csv",
)
CLASSICAL_BEST_PATH = os.path.join(
    OUT_DIR,
    "combined_classical_best_per_condition_with_std.csv",
)

FAST_DL_OUT_DIR = os.path.join(
    OUT_DIR,
    "DL_NO_ELAPSED_ALL_ARCHITECTURES",
)
DL_SUMMARY_PATH = os.path.join(
    FAST_DL_OUT_DIR,
    "combined_dl_no_elapsed_summary_with_std.csv",
)
DL_FOLDS_PATH = os.path.join(
    FAST_DL_OUT_DIR,
    "combined_dl_no_elapsed_fold_metrics.csv",
)
DL_BEST_PATH = os.path.join(
    FAST_DL_OUT_DIR,
    "combined_dl_no_elapsed_best_per_task_sensor_with_std.csv",
)

if "combined_classical_best" not in globals():
    if all(
        os.path.exists(path)
        for path in [
            CLASSICAL_SUMMARY_PATH,
            CLASSICAL_FOLDS_PATH,
            CLASSICAL_BEST_PATH,
        ]
    ):
        print("Reloading completed classical results.")
        combined_classical = pd.read_csv(CLASSICAL_SUMMARY_PATH)
        combined_classical_folds = pd.read_csv(CLASSICAL_FOLDS_PATH)
        combined_classical_best = pd.read_csv(CLASSICAL_BEST_PATH)

if "combined_fast_dl_best" not in globals():
    if all(
        os.path.exists(path)
        for path in [
            DL_SUMMARY_PATH,
            DL_FOLDS_PATH,
            DL_BEST_PATH,
        ]
    ):
        print("Reloading completed DL results.")
        combined_fast_dl = pd.read_csv(DL_SUMMARY_PATH)
        combined_fast_folds = pd.read_csv(DL_FOLDS_PATH)
        combined_fast_dl_best = pd.read_csv(DL_BEST_PATH)


required_classical = {
    "combined_classical_best",
    "combined_classical_folds",
}
required_dl = {
    "combined_fast_dl_best",
    "combined_fast_folds",
}

missing = [
    name
    for name in required_classical | required_dl
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "Run the classical and DL cells first. "
        f"Missing variables: {missing}"
    )

long_rows = []

for _, row in combined_classical_best.iterrows():
    regime = (
        "classical_no_elapsed"
        if row["time_condition"] == "no_elapsed"
        else "classical_with_elapsed"
    )

    long_rows.append({
        "task": row["task"],
        "sensor_combo": row["sensor_combo"],
        "regime": regime,
        "model_family": "classical",
        "best_model": row["model"],
        "selection": f"k={row['k']}",
        "n_features": row["n_features"],
        "pooled_accuracy": row["accuracy"],
        "pooled_macro_f1": row["macro_f1"],
        "pooled_balanced_accuracy": (
            row["balanced_accuracy"]
        ),
        "fold_accuracy_mean": (
            row["fold_accuracy_mean"]
        ),
        "fold_accuracy_std": (
            row["fold_accuracy_std"]
        ),
        "fold_macro_f1_mean": (
            row["fold_macro_f1_mean"]
        ),
        "fold_macro_f1_std": (
            row["fold_macro_f1_std"]
        ),
        "fold_balanced_accuracy_mean": (
            row["fold_balanced_accuracy_mean"]
        ),
        "fold_balanced_accuracy_std": (
            row["fold_balanced_accuracy_std"]
        ),
        "n_folds": row["n_folds"],
    })

for _, row in combined_fast_dl_best.iterrows():
    long_rows.append({
        "task": row["task"],
        "sensor_combo": row["sensor_combo"],
        "regime": "dl_no_elapsed",
        "model_family": "DL",
        "best_model": row["model_type"],
        "selection": (
            f"seq={int(row['seq_len'])}, "
            f"k={int(row['k_features'])}"
        ),
        "n_features": (
            row["n_features_before_select"]
        ),
        "pooled_accuracy": row["accuracy"],
        "pooled_macro_f1": row["macro_f1"],
        "pooled_balanced_accuracy": (
            row["balanced_accuracy"]
        ),
        "fold_accuracy_mean": (
            row["fold_accuracy_mean"]
        ),
        "fold_accuracy_std": (
            row["fold_accuracy_std"]
        ),
        "fold_macro_f1_mean": (
            row["fold_macro_f1_mean"]
        ),
        "fold_macro_f1_std": (
            row["fold_macro_f1_std"]
        ),
        "fold_balanced_accuracy_mean": (
            row["fold_balanced_accuracy_mean"]
        ),
        "fold_balanced_accuracy_std": (
            row["fold_balanced_accuracy_std"]
        ),
        "n_folds": row["n_folds"],
    })

publication_long = pd.DataFrame(long_rows)

publication_long["accuracy_mean_pm_std"] = (
    publication_long.apply(
        lambda row: (
            f"{row['fold_accuracy_mean']:.3f} "
            f"± {row['fold_accuracy_std']:.3f}"
        ),
        axis=1,
    )
)
publication_long["macro_f1_mean_pm_std"] = (
    publication_long.apply(
        lambda row: (
            f"{row['fold_macro_f1_mean']:.3f} "
            f"± {row['fold_macro_f1_std']:.3f}"
        ),
        axis=1,
    )
)
publication_long[
    "balanced_accuracy_mean_pm_std"
] = publication_long.apply(
    lambda row: (
        f"{row['fold_balanced_accuracy_mean']:.3f} "
        f"± {row['fold_balanced_accuracy_std']:.3f}"
    ),
    axis=1,
)

regime_order = [
    "classical_no_elapsed",
    "classical_with_elapsed",
    "dl_no_elapsed",
]
publication_long["regime"] = pd.Categorical(
    publication_long["regime"],
    categories=regime_order,
    ordered=True,
)
publication_long = publication_long.sort_values(
    ["task", "sensor_combo", "regime"]
).reset_index(drop=True)

publication_long.to_csv(
    os.path.join(
        OUT_DIR,
        "publication_three_regime_comparison_long.csv",
    ),
    index=False,
)

display_columns = [
    "task",
    "sensor_combo",
    "regime",
    "best_model",
    "selection",
    "pooled_accuracy",
    "pooled_macro_f1",
    "pooled_balanced_accuracy",
    "accuracy_mean_pm_std",
    "macro_f1_mean_pm_std",
    "balanced_accuracy_mean_pm_std",
]
display(
    publication_long[display_columns]
    .round(4)
)

# Compact wide table resembling the report.
compact = publication_long.copy()
compact["result_text"] = compact.apply(
    lambda row: (
        f"{row['best_model']} ({row['selection']}); "
        f"pooled A={row['pooled_accuracy']:.3f}, "
        f"M={row['pooled_macro_f1']:.3f}, "
        f"B={row['pooled_balanced_accuracy']:.3f}; "
        f"fold A={row['accuracy_mean_pm_std']}, "
        f"M={row['macro_f1_mean_pm_std']}"
    ),
    axis=1,
)

publication_wide = (
    compact.pivot(
        index=["task", "sensor_combo"],
        columns="regime",
        values="result_text",
    )
    .reset_index()
)

winner_rows = (
    publication_long.sort_values(
        [
            "task",
            "sensor_combo",
            "pooled_macro_f1",
            "pooled_accuracy",
        ],
        ascending=[
            True,
            True,
            False,
            False,
        ],
    )
    .groupby(
        ["task", "sensor_combo"],
        as_index=False,
    )
    .head(1)
    [[
        "task",
        "sensor_combo",
        "regime",
        "best_model",
        "pooled_macro_f1",
    ]]
    .rename(
        columns={
            "regime": "best_regime_by_pooled_macro_f1",
            "best_model": "best_model_overall",
            "pooled_macro_f1": (
                "best_pooled_macro_f1"
            ),
        }
    )
)

publication_wide = publication_wide.merge(
    winner_rows,
    on=["task", "sensor_combo"],
    how="left",
)

publication_wide.to_csv(
    os.path.join(
        OUT_DIR,
        "publication_three_regime_comparison_wide.csv",
    ),
    index=False,
)

display(publication_wide)

print("Saved all final tables to:")
print(OUT_DIR)


,task,sensor_combo,regime,best_model,selection,pooled_accuracy,pooled_macro_f1,pooled_balanced_accuracy,accuracy_mean_pm_std,macro_f1_mean_pm_std,balanced_accuracy_mean_pm_std
0,conversation_vs_building,OE,classical_no_elapsed,logreg_C1,k=200,0.7008,0.6877,0.6974,0.673 ± 0.206,0.603 ± 0.186,0.625 ± 0.126
1,conversation_vs_building,OE,classical_with_elapsed,logreg_C1,k=80,0.7835,0.7681,0.7712,0.748 ± 0.171,0.645 ± 0.202,0.682 ± 0.131
2,conversation_vs_building,OE,dl_no_elapsed,transformer,"seq=9, k=120",0.7560,0.6972,0.6889,0.725 ± 0.177,0.611 ± 0.178,0.657 ± 0.127
3,conversation_vs_building,OE_OPTI,classical_no_elapsed,logreg_C1,k=80,0.8161,0.8033,0.8072,0.809 ± 0.061,0.732 ± 0.127,0.750 ± 0.110
4,conversation_vs_building,OE_OPTI,classical_with_elapsed,logreg_C1,k=80,0.8324,0.8216,0.8276,0.813 ± 0.104,0.727 ± 0.159,0.742 ± 0.132
5,conversation_vs_building,OE_OPTI,dl_no_elapsed,bilstm,"seq=9, k=120",0.8806,0.8591,0.8591,0.870 ± 0.070,0.773 ± 0.141,0.774 ± 0.127
6,conversation_vs_building,OE_OPTI_XSENS,classical_no_elapsed,logreg_C1,k=80,0.8161,0.8033,0.8072,0.809 ± 0.061,0.732 ± 0.127,0.750 ± 0.110
7,conversation_vs_building,OE_OPTI_XSENS,classical_with_elapsed,logreg_C1,k=80,0.8324,0.8216,0.8276,0.813 ± 0.104,0.727 ± 0.159,0.742 ± 0.132
8,conversation_vs_building,OE_OPTI_XSENS,dl_no_elapsed,transformer,"seq=9, k=120",0.8806,0.8549,0.8451,0.868 ± 0.101,0.720 ± 0.222,0.736 ± 0.191
9,conversation_vs_building,OE_XSENS,classical_no_elapsed,linearSVC_C1,k=80,0.6240,0.6154,0.6309,0.606 ± 0.218,0.543 ± 0.207,0.627 ± 0.141


,task,sensor_combo,classical_no_elapsed,classical_with_elapsed,dl_no_elapsed,best_regime_by_pooled_macro_f1,best_model_overall,best_pooled_macro_f1
0,conversation_vs_building,OE,"logreg_C1 (k=200); pooled A=0.701, M=0.688, B=0.697; fold A=0.673 ± 0.206, M=0.603 ± 0.186","logreg_C1 (k=80); pooled A=0.783, M=0.768, B=0.771; fold A=0.748 ± 0.171, M=0.645 ± 0.202","transformer (seq=9, k=120); pooled A=0.756, M=0.697, B=0.689; fold A=0.725 ± 0.177, M=0.611 ± 0.178",classical_with_elapsed,logreg_C1,0.768121
1,conversation_vs_building,OE_OPTI,"logreg_C1 (k=80); pooled A=0.816, M=0.803, B=0.807; fold A=0.809 ± 0.061, M=0.732 ± 0.127","logreg_C1 (k=80); pooled A=0.832, M=0.822, B=0.828; fold A=0.813 ± 0.104, M=0.727 ± 0.159","bilstm (seq=9, k=120); pooled A=0.881, M=0.859, B=0.859; fold A=0.870 ± 0.070, M=0.773 ± 0.141",dl_no_elapsed,bilstm,0.859122
2,conversation_vs_building,OE_OPTI_XSENS,"logreg_C1 (k=80); pooled A=0.816, M=0.803, B=0.807; fold A=0.809 ± 0.061, M=0.732 ± 0.127","logreg_C1 (k=80); pooled A=0.832, M=0.822, B=0.828; fold A=0.813 ± 0.104, M=0.727 ± 0.159","transformer (seq=9, k=120); pooled A=0.881, M=0.855, B=0.845; fold A=0.868 ± 0.101, M=0.720 ± 0.222",dl_no_elapsed,transformer,0.854890
3,conversation_vs_building,OE_XSENS,"linearSVC_C1 (k=80); pooled A=0.624, M=0.615, B=0.631; fold A=0.606 ± 0.218, M=0.543 ± 0.207","linearSVC_C1 (k=80); pooled A=0.735, M=0.717, B=0.721; fold A=0.713 ± 0.160, M=0.630 ± 0.163","transformer (seq=9, k=120); pooled A=0.719, M=0.648, B=0.641; fold A=0.664 ± 0.278, M=0.518 ± 0.230",classical_with_elapsed,linearSVC_C1,0.717186
4,conversation_vs_building,OPTI,"logreg_C1 (k=80); pooled A=0.830, M=0.817, B=0.819; fold A=0.829 ± 0.061, M=0.752 ± 0.140","logreg_C1 (k=80); pooled A=0.847, M=0.838, B=0.845; fold A=0.827 ± 0.101, M=0.742 ± 0.166","lstm (seq=9, k=120); pooled A=0.875, M=0.852, B=0.850; fold A=0.861 ± 0.100, M=0.739 ± 0.205",dl_no_elapsed,lstm,0.852079
5,conversation_vs_building,OPTI_XSENS,"logreg_C1 (k=80); pooled A=0.830, M=0.817, B=0.819; fold A=0.829 ± 0.061, M=0.752 ± 0.140","logreg_C1 (k=80); pooled A=0.847, M=0.838, B=0.845; fold A=0.827 ± 0.101, M=0.742 ± 0.166","bilstm (seq=9, k=120); pooled A=0.881, M=0.860, B=0.863; fold A=0.868 ± 0.103, M=0.748 ± 0.205",dl_no_elapsed,bilstm,0.860094
6,conversation_vs_building,XSENS,"logreg_C1 (k=80); pooled A=0.644, M=0.634, B=0.649; fold A=0.621 ± 0.185, M=0.562 ± 0.191","logreg_C1 (k=80); pooled A=0.719, M=0.715, B=0.740; fold A=0.694 ± 0.229, M=0.616 ± 0.246","lstm (seq=9, k=120); pooled A=0.652, M=0.615, B=0.627; fold A=0.628 ± 0.124, M=0.534 ± 0.140",classical_with_elapsed,logreg_C1,0.714589
7,conversation_vs_merging,OE,"logreg_C1 (k=80); pooled A=0.739, M=0.707, B=0.723; fold A=0.765 ± 0.163, M=0.674 ± 0.169","logreg_C1 (k=200); pooled A=0.797, M=0.768, B=0.780; fold A=0.819 ± 0.153, M=0.700 ± 0.156","bilstm (seq=9, k=120); pooled A=0.664, M=0.658, B=0.680; fold A=0.707 ± 0.187, M=0.630 ± 0.194",classical_with_elapsed,logreg_C1,0.767907
8,conversation_vs_merging,OE_OPTI,"logreg_C1 (k=80); pooled A=0.822, M=0.791, B=0.796; fold A=0.794 ± 0.147, M=0.710 ± 0.221","logreg_C1 (k=80); pooled A=0.860, M=0.835, B=0.838; fold A=0.855 ± 0.094, M=0.755 ± 0.210","transformer (seq=9, k=120); pooled A=0.879, M=0.866, B=0.859; fold A=0.898 ± 0.063, M=0.775 ± 0.190",dl_no_elapsed,transformer,0.865752
9,conversation_vs_merging,OE_OPTI_XSENS,"logreg_C1 (k=80); pooled A=0.822, M=0.791, B=0.796; fold A=0.794 ± 0.147, M=0.710 ± 0.221","logreg_C1 (k=80); pooled A=0.860, M=0.835, B=0.838; fold A=0.855 ± 0.094, M=0.755 ± 0.210","transformer (seq=9, k=120); pooled A=0.874, M=0.861, B=0.857; fold A=0.896 ± 0.090, M=0.828 ± 0.178",dl_no_elapsed,transformer,0.860813


Saved all final tables to:
/content/drive/MyDrive/thesis/data/PUBLICATION_TASK2_FULL_COMPARISON
